# Unit 14: Compare and select policies

Build an exact evaluation case list, join paired outcomes, and include every attempt. Use final states only after Unit 13 and after recipes are frozen.

**Notebook:** learner copy. Work through the cells in order.

In [ ]:
from __future__ import annotations


from typing import TYPE_CHECKING

if TYPE_CHECKING:
    from collections.abc import Mapping
    from collections.abc import Sequence
    from vla_lab.teaching.records import CaseResult
    from vla_lab.teaching.records import PairedSummary


from pathlib import Path
import os, sys
WORKSPACE = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'manifests/act_references.yaml').is_file()), None)
if WORKSPACE is None:
    raise FileNotFoundError('Run this notebook inside the VLA course workspace')
sys.path.insert(0, str(WORKSPACE / 'src'))
HF_ROOT = WORKSPACE / '.local-data' / 'hf'
os.environ.setdefault('HF_HUB_CACHE', str(HF_ROOT / 'hub'))
os.environ.setdefault('HF_DATASETS_CACHE', str(HF_ROOT / 'datasets'))
os.environ.setdefault('HF_HUB_OFFLINE', '1')
os.environ.setdefault('HF_DATASETS_OFFLINE', '1')
print('Workspace:', WORKSPACE)


In [ ]:
from __future__ import annotations
from vla_lab.teaching.evaluation import development_cases, final_cases, validate_case_results, paired_summary, evaluate_frozen_method_timed


## Exercise 1: Select development cases by purpose

**Purpose:** Use separate states for exploration, method checks, and data intervention.

**Contract:** `development_cases(stage) -> list[int]`

`exploration` returns 0–9; `method` returns 10–14; `intervention` returns 15–19.

**Method:** Keep these groups fixed so experiments do not use the final set.

**Pointers:** Try the state IDs once and inspect the exact returned list.

<details><summary>Detailed hint</summary>

A dictionary maps each stage name to one range.

</details>

Estimated time: about 15 minutes.

In [ ]:
def development_cases(stage: str) -> list[int]:
    raise NotImplementedError('Exercise 1: select the declared development group')

### Check

In [ ]:
assert development_cases('method') == [10, 11, 12, 13, 14], 'Method states are wrong'

### Use your function

In [ ]:
print('Method check states:', development_cases('method'))

## Exercise 2: Check complete evaluation records

**Purpose:** Catch missing or duplicate cases before reporting a score.

**Contract:** `validate_case_results(rows, *, expected_case_ids) -> ordered rows`

Each row has a unique `case_id`. Require exact coverage and return rows in expected order.

**Method:** Compare actual IDs with the frozen expected set, then order records by ID.

**Pointers:** Never silently drop a failed or missing run.

<details><summary>Detailed hint</summary>

Build a map only after checking duplicate IDs.

</details>

Estimated time: about 15 minutes.

In [ ]:
def validate_case_results(
    results: Sequence[CaseResult], *, expected_case_ids: Sequence[str | int]
) -> list[CaseResult]:
    raise NotImplementedError('Exercise 2: validate exact case coverage')

### Check

In [ ]:
ordered = validate_case_results([{'case_id': 1, 'success': False}, {'case_id': 0, 'success': True}], expected_case_ids=[0, 1])
assert [row['case_id'] for row in ordered] == [0, 1], 'Rows are not in frozen case order'

### Use your function

In [ ]:
print('Included case IDs:', [row['case_id'] for row in ordered])

## Exercise 3: Summarize paired attempts

**Purpose:** Compare two recipes on the same cases and report disagreements.

**Contract:** `paired_summary(left, right, *, expected_case_ids) -> summary`

Pair by `case_id = state_id + reset_seed`. Each row also carries task, partition, prompt, action length, policy revision, and success. State, seed, task, partition, and prompt must match. Keep each policy's action length and revision fixed across its rows.

**Method:** Pair records before computing rates and disagreements. Count only explicit `success=True` as success; retain all failed attempts in the denominator.

**Pointers:** Small paired samples can be inconclusive; show disagreement counts.

<details><summary>Detailed hint</summary>

Count `left_success and not right_success` separately from the reverse. Check state ID and reset seed as well as the display case key.

</details>

Estimated time: about 15 minutes.

In [ ]:
def paired_summary(
    left: Sequence[CaseResult],
    right: Sequence[CaseResult],
    *,
    expected_case_ids: Sequence[str | int],
) -> PairedSummary:
    raise NotImplementedError('Exercise 3: aggregate paired attempts')

### Check

In [ ]:
base = {'state_id': 0, 'reset_seed': 4, 'task_id': 'bowl_plate', 'partition': 'method', 'prompt': 'place bowl', 'action_steps': 10}
left = [{**base, 'case_id': 'state-0-seed-4', 'policy_revision': 'rev-a', 'success': True}]
right = [{**base, 'case_id': 'state-0-seed-4', 'policy_revision': 'rev-b', 'success': False}]
summary = paired_summary(left, right, expected_case_ids=['state-0-seed-4'])
assert summary['cases'] == 1 and summary['left_only'] == 1 and summary['right_only'] == 0, 'Paired disagreement counts are wrong'

### Use your function

In [ ]:
print('Paired difference:', summary['paired_difference'], 'disagreements:', summary['left_only'], summary['right_only'])

## Exercise 4: Open the final evaluation set

**Purpose:** Protect the final states from earlier model and recipe choices.

**Contract:** `final_cases(*, data_unit_complete, recipes_frozen) -> list[int]`

Return states 20–49 only when Unit 13 data work is complete and all recipes are frozen.

**Method:** Require both gates before returning the final IDs.

**Pointers:** A valid final comparison can be inconclusive. Make no changes after seeing these outcomes.

<details><summary>Detailed hint</summary>

Check both booleans before creating the range.

</details>

Estimated time: about 15 minutes.

In [ ]:
def final_cases(*, data_unit_complete: bool, recipes_frozen: bool) -> list[int]:
    raise NotImplementedError('Exercise 4: enforce the final-set gate')

### Check

In [ ]:
try:
    final_cases(data_unit_complete=False, recipes_frozen=True)
except ValueError:
    pass
else:
    raise AssertionError('Final states opened before the data unit was complete')

### Use your function

In [ ]:
print('Final set size:', len(final_cases(data_unit_complete=True, recipes_frozen=True)))

## Larger experiment

Freeze the task, checkpoint hashes, prompt, action length, reset seed rule, and time budget before final evaluation. Copy the pinned comparison recipe to a new output folder; verify absolute checkpoint paths and on-disk hashes before evaluation. Use states 20–49 once for each recipe. The evaluator records every policy or infrastructure failure. Pair by `(state_id, reset_seed)` and report all attempts and disagreements. It records both action-selection API latency (which includes queued actions) and action-generation latency. Report mean, p95, maximum, and every attempt; missing timing stays `None`. This is a comparison of prepared practical recipes: training budgets and pretraining differ. Report those recipe fields so the table does not imply a from-scratch or equal-training comparison.

Set `RUN_FINAL=False` by default because the full comparison runs 90 simulator episodes. The runner uses local checkpoints and does not download. Keep outputs in a new folder. Do not report historical results as new runs. The code resolves checkpoint paths against the workspace, checks hashes before the first episode, writes every attempt, a table, and a plot. The table includes checkpoint hash, robot training steps, and pretraining provenance from the frozen recipe.

In [ ]:
RUN_FINAL = False
if RUN_FINAL:
    import json
    import hashlib
    import yaml
    from pathlib import Path
    import matplotlib.pyplot as plt
    from vla_lab.evaluation.comparison import METHODS, load_comparison_config
    from vla_lab.teaching.live import find_workspace
    root = find_workspace()
    output = root / '.local-data/course-results/final-bowl-plate-course-01'
    output.mkdir(parents=True, exist_ok=False)
    frozen_config = output / 'comparison.yaml'
    recipe = yaml.safe_load((root / 'configs/experiments/comparison.yaml').read_text())
    for method, item in recipe['policies'].items():
        checkpoint = Path(item['checkpoint'])
        item['checkpoint'] = str((root / checkpoint).resolve() if not checkpoint.is_absolute() else checkpoint.resolve())
        if 'vlm_dir' in item:
            vlm_dir = Path(item['vlm_dir'])
            item['vlm_dir'] = str((root / vlm_dir).resolve() if not vlm_dir.is_absolute() else vlm_dir.resolve())
    frozen_config.write_text(yaml.safe_dump(recipe, sort_keys=False))
    config = load_comparison_config(frozen_config, root / 'configs')
    def sha256_file(path: Path) -> str:
        digest = hashlib.sha256()
        with path.open('rb') as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(block)
        return digest.hexdigest()
    for method, item in config['policies'].items():
        checkpoint = Path(item['checkpoint'])
        weights = checkpoint / 'model.safetensors'
        if not checkpoint.is_absolute() or not weights.is_file():
            raise FileNotFoundError(f'{method} checkpoint is missing or not absolute: {checkpoint}')
        if sha256_file(weights) != item['model_sha256']:
            raise ValueError(f'{method} checkpoint hash differs from the frozen recipe')
        if 'vlm_dir' in item and not Path(item['vlm_dir']).is_dir():
            raise FileNotFoundError(f"SmolVLA VLM directory is missing: {item['vlm_dir']}")
    # This gate runs only after the frozen recipe and each local model hash pass.
    recipes_frozen = True
    recovery = root / '.local-data/course-results/recovery'
    paired_path = recovery / 'paired-summary.json'
    provenance_path = recovery / 'continued/recovery_provenance.json'
    baseline_summary_path = recovery / 'eval-baseline/summary.json'
    changed_summary_path = recovery / 'eval-changed/summary.json'
    data_complete = all(path.is_file() for path in (
        paired_path, provenance_path, baseline_summary_path, changed_summary_path))
    if data_complete:
        paired_data = json.loads(paired_path.read_text())
        baseline_eval = json.loads(baseline_summary_path.read_text())
        changed_eval = json.loads(changed_summary_path.read_text())
        data_complete = (
            paired_data.get('cases') == 5
            and paired_data.get('failed_attempts_included') is True
            and baseline_eval.get('attempts') == 5
            and changed_eval.get('attempts') == 5
        )
    final_ids = final_cases(data_unit_complete=data_complete, recipes_frozen=True)
    assert final_ids == config['state_ids']
    result_dirs = {method: output / method for method in METHODS}
    timing_by_method = {}
    for method in METHODS:
        timed_result = evaluate_frozen_method_timed(method=method, config_path=str(frozen_config),
            configs_root=root / 'configs', data_root=root / '.local-data',
            output_dir=result_dirs[method])
        method_summary = timed_result['summary']
        timing_by_method[method] = timed_result['timing']
        print(method, 'attempts:', method_summary['attempts'], 'successes:', method_summary['successes'],
              'policy/infrastructure errors:', method_summary['policy_errors'], method_summary['infrastructure_errors'])
    rows_by_method = {}
    table = []
    for method in METHODS:
        saved = [json.loads(line) for line in (result_dirs[method] / 'episodes.jsonl').read_text().splitlines()]
        recipe = config['policies'][method]
        converted = []
        for row in saved:
            seed = int(row['metrics']['seed'])
            state_id = int(row['initial_state_id'])
            converted.append({'case_id': f'state-{state_id}-seed-{seed}', 'state_id': state_id,
                'reset_seed': seed, 'task_id': row['task_id'], 'partition': row['partition'],
                'prompt': config['task_prompt'], 'action_steps': recipe['n_action_steps'],
                'policy_revision': row['policy_revision'], 'success': row['success'] is True})
        rows_by_method[method] = converted
        successes = sum(row['success'] for row in converted)
        elapsed = [float(row['metrics']['elapsed_s']) for row in saved if row['metrics'].get('elapsed_s') is not None]
        action_api = timing_by_method[method]['action_selection_api']
        action_generation = timing_by_method[method]['action_generation']
        table.append({'method': method, 'successes': successes, 'attempts': len(converted),
            'success_rate': successes / len(converted),
            'recipe_file': str(frozen_config), 'checkpoint_sha256': recipe['model_sha256'],
            'robot_training_steps': recipe.get('robot_training_steps'),
            'pretraining': recipe.get('pretraining'),
            'mean_episode_seconds': (sum(elapsed) / len(elapsed)) if elapsed else None,
            'mean_action_api_seconds': action_api['mean_s'], 'p95_action_api_seconds': action_api['p95_s'],
            'max_action_api_seconds': action_api['max_s'], 'action_api_calls': action_api['count'],
            'mean_action_generation_seconds': action_generation['mean_s'],
            'p95_action_generation_seconds': action_generation['p95_s'],
            'max_action_generation_seconds': action_generation['max_s'],
            'action_generation_calls': action_generation['count'],
            'timing_scope': 'episode wall time includes simulator and processing; action API includes queue reads; action generation is the model chunk hook',
            'policy_errors': sum(row['termination_reason'] == 'policy_error' for row in saved),
            'infrastructure_errors': sum(row['termination_reason'] == 'infrastructure_error' for row in saved)})
    expected = [f"state-{state}-seed-{config['base_seed'] + state}" for state in final_ids]
    for left_name, right_name in ((METHODS[0], METHODS[1]), (METHODS[0], METHODS[2]), (METHODS[1], METHODS[2])):
        pair = paired_summary(rows_by_method[left_name], rows_by_method[right_name], expected_case_ids=expected)
        print(left_name, 'vs', right_name, pair)
    (output / 'course-table.json').write_text(json.dumps(table, indent=2) + '\n')
    plt.bar([item['method'] for item in table], [item['success_rate'] for item in table])
    plt.ylim(0, 1)
    plt.ylabel('Success rate over all attempts')
    plt.tight_layout()
    plt.savefig(output / 'success-rates.png', dpi=160)
    print('Saved recipe, all attempts, table, and plot under', output)

## Record your result

Keep the inputs, every attempt, and the result table with the notebook. A failed run is a result and stays in the denominator.